# V18 C3 Gene Count Verification
## 목적: "169" vs "173" vs "196" 불일치 해소

**확인 항목:**
1. C3 코드에 정의된 후보 유전자 수 (중복 제거)
2. h5ad 파일에 실제 존재하는 유전자 수
3. C3_gene_list CSV 파일 내용
4. C5 148-gene panel과의 차이
5. C4 pathway 수 (26 vs 29)
6. C9/C9B 파일 확인

**방어 코드:** donor/lineage/sample 컬럼 자동감지 포함

**Date:** 2026-03-06

In [ ]:
# ============================================================
# CELL 1: Mount Drive + Imports
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# DEFENSIVE HELPER FUNCTIONS (재사용 가능)
# ============================================================

def find_column(df, candidates, required=False, label='column'):
    """Flexible column auto-detection across 다양한 이름 변형.
    Returns column name if found, None otherwise.
    If required=True, raises clear error with available columns."""
    for c in candidates:
        if c in df.columns:
            return c
    # Case-insensitive fallback
    col_lower = {col.lower(): col for col in df.columns}
    for c in candidates:
        if c.lower() in col_lower:
            return col_lower[c.lower()]
    if required:
        raise KeyError(
            f"'{label}' column not found. "
            f"Tried: {candidates}. "
            f"Available: {list(df.columns)}"
        )
    return None

# Column name candidates for common fields
DONOR_COLS = ['donor', 'donor_id', 'sample_id', 'orig.ident',
              'patient', 'subject', 'Patient', 'Donor', 'sample',
              'donor_name', 'SampleID', 'DonorID']

LINEAGE_COLS = ['lineage', 'cell_type', 'celltype', 'Lineage',
                'CellType', 'cell_lineage', 'lineage_annotation',
                'major_celltype', 'broad_type']

GENE_COLS = ['gene', 'Gene', 'gene_name', 'GeneName', 'genes',
             'feature', 'symbol', 'gene_symbol', 'Gene_Symbol']

PATHWAY_COLS = ['pathway', 'Pathway', 'gene_set', 'geneset',
                'GeneSet', 'pathway_name', 'set_name']

TISSUE_COLS = ['tissue', 'Tissue', 'compartment', 'tissue_type']

COMPARISON_COLS = ['comparison', 'Comparison', 'comp', 'contrast', 'group_comparison']

def safe_read_csv(path, label='file'):
    """Read CSV with error handling."""
    if not os.path.exists(path):
        print(f"  ⚠️ {label}: File not found → {path}")
        return None
    try:
        df = pd.read_csv(path)
        print(f"  ✅ {label}: {df.shape[0]} rows × {df.shape[1]} cols")
        return df
    except Exception as e:
        print(f"  ❌ {label}: Read error → {e}")
        return None

def extract_genes_from_df(df, label='DataFrame'):
    """Try to extract gene names from a DataFrame, checking multiple column candidates."""
    # Try known gene column names
    gcol = find_column(df, GENE_COLS)
    if gcol:
        genes = set(df[gcol].dropna().astype(str).unique())
        print(f"    → {len(genes)} genes from column '{gcol}'")
        return genes
    # Fallback: check first column for gene-like strings
    first_col = df.columns[0]
    vals = df[first_col].dropna().astype(str).unique()
    gene_like = [v for v in vals if 1 < len(v) < 20 and not v.replace('.','').replace('-','').isdigit()]
    if len(gene_like) > 10:
        genes = set(gene_like)
        print(f"    → {len(genes)} gene-like values from first column '{first_col}'")
        return genes
    print(f"    → No gene column found. Columns: {list(df.columns)[:8]}")
    return set()

print("✅ Helper functions loaded")

In [ ]:
# ============================================================
# CELL 2: Define ALL gene lists (원본 코드 그대로)
# ============================================================
print("=" * 70)
print("  STEP 1: Gene list definitions from C3/C3B code")
print("=" * 70)

C3_GENES = [
    'AIM2', 'NLRC4', 'MEFV', 'CASP1', 'NLRP3', 'IL1B', 'GSDMD', 'PYCARD',
    'CASP4', 'CASP5', 'NAIP',
    'TLR2', 'TLR4', 'TLR7', 'TLR8', 'TLR9', 'MYD88', 'TICAM1',
    'DDX58', 'IFIH1', 'MAVS', 'CGAS', 'STING1',
    'IFNA1', 'IFNB1', 'IFNAR1', 'IFNAR2', 'STAT1', 'STAT2', 'IRF3', 'IRF7',
    'IRF9', 'MX1', 'MX2', 'OAS1', 'ISG15', 'IFIT1',
    'LGALS9', 'TGFB1', 'IL10', 'IDO1', 'TNFAIP3', 'HAVCR2', 'CD274',
    'PDCD1LG2', 'VSIR', 'SIGLEC10', 'LILRB1', 'LILRB2',
    'FOXP3', 'IL2RA', 'CTLA4', 'IKZF2', 'TNFRSF18', 'TIGIT',
    'ENTPD1', 'NT5E', 'LRRC32', 'BACH2', 'PRDM1', 'IL7R',
    'PDCD1', 'LAG3', 'TOX', 'EOMES', 'BATF', 'NFATC1',
    'TBX21', 'TCF7', 'SLAMF6', 'CXCR5', 'GZMB',
    'ENTPD1', 'HAVCR2', 'CX3CR1', 'PRF1', 'GNLY', 'FGFBP2',
    'XCL1', 'XCL2', 'SELL',
    'GZMA', 'GZMK', 'GZMH', 'GZMM', 'NKG7', 'KLRK1', 'KLRD1',
    'NCR1', 'NCR3', 'FCGR3A', 'CD160', 'KIR2DL4',
    'MTOR', 'RPTOR', 'RICTOR', 'RPS6KB1', 'EIF4EBP1', 'AKT1',
    'HIF1A', 'LDHA', 'PKM', 'SLC2A1', 'PFKFB3',
    'NDUFS1', 'COX5A', 'ATP5F1A', 'UQCRC1', 'SDHB',
    'CPT1A', 'ACADVL', 'HADHA', 'PPARGC1A',
    'JAK1', 'JAK2', 'JAK3', 'TYK2', 'STAT3', 'STAT4', 'STAT5A',
    'STAT5B', 'STAT6', 'SOCS1', 'SOCS3', 'CISH', 'PIAS1',
    'CD19', 'MS4A1', 'CD79A', 'CD79B', 'PAX5', 'BCL6',
    'IRF4', 'XBP1', 'SDC1', 'TNFRSF17', 'MZB1',
    'BCL2', 'MCL1', 'BCL2L1', 'BIRC3', 'CFLAR',
    'BAX', 'BAK1', 'BID', 'BBC3', 'CASP3', 'CASP8', 'FAS', 'FASLG',
    'TP53', 'ATM', 'ATR', 'BRCA1', 'CHEK1', 'CHEK2',
    'DNMT1', 'DNMT3A', 'TET2', 'HDAC1', 'EZH2', 'KDM6A',
    'CCR7', 'CXCR3', 'CXCR4', 'CXCR6', 'CCR2', 'CCR5',
    'CCL3', 'CCL4', 'CCL5', 'CXCL10', 'CXCL13', 'CX3CR1',
    'HLA-A', 'HLA-B', 'HLA-C', 'HLA-DRA', 'HLA-DRB1', 'HLA-DPA1',
    'HLA-DPB1', 'B2M', 'TAP1', 'TAP2', 'CIITA', 'CD74',
    'TERT', 'MYC', 'VEGFA', 'HGF', 'MET', 'CTNNB1',
    'APC', 'AXIN1', 'TP53', 'RB1', 'CDKN2A', 'MDM2',
]

C3B_GENES = [
    'AICDA', 'JCHAIN', 'IL1RN', 'CD27', 'TOX2',
    'RICTOR', 'AIM2', 'MEFV', 'NLRC4', 'CASP1', 'LGALS9', 'TGFB1',
    'MTOR', 'JAK1', 'RPTOR', 'PYCARD', 'PRDM1',
]

# Dedup analysis
from collections import Counter
c3_set = set(C3_GENES)
c3b_set = set(C3B_GENES)
c3b_truly_new = c3b_set - c3_set
ALL_CANDIDATES = c3_set | c3b_set
dups_in_c3 = {k:v for k,v in Counter(C3_GENES).items() if v > 1}

print(f"C3 raw list: {len(C3_GENES)} entries → {len(c3_set)} unique")
print(f"  Duplicates within C3: {dups_in_c3}")
print(f"C3B: {len(C3B_GENES)} entries → {len(c3b_set)} unique")
print(f"  C3B truly new: {len(c3b_truly_new)} → {sorted(c3b_truly_new)}")
print(f"ALL CANDIDATES (C3∪C3B): {len(ALL_CANDIDATES)} unique")

In [ ]:
# ============================================================
# CELL 3: Load h5ad → check gene availability
# ============================================================
import scanpy as sc

print("=" * 70)
print("  STEP 2: Load h5ad and filter genes")
print("=" * 70)

DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'

if not os.path.exists(DATA_PATH):
    print(f"❌ h5ad not found: {DATA_PATH}")
    # Try alternatives
    alt_base = '/content/drive/MyDrive/ITLAS/data/'
    if os.path.exists(alt_base):
        for dirpath, _, files in os.walk(alt_base):
            for f in files:
                if f.endswith('.h5ad'):
                    print(f"  Found: {os.path.join(dirpath, f)}")
else:
    adata = sc.read_h5ad(DATA_PATH)
    print(f"✅ h5ad loaded: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")
    
    # Available columns in obs
    print(f"\nobs columns ({len(adata.obs.columns)}): {list(adata.obs.columns)}")
    
    # Detect key columns
    donor_col = find_column(adata.obs, DONOR_COLS, required=False, label='donor')
    lineage_col = find_column(adata.obs, LINEAGE_COLS, required=False, label='lineage')
    tissue_col = find_column(adata.obs, TISSUE_COLS, required=False, label='tissue')
    
    print(f"\n  Detected donor column:   {donor_col}")
    if donor_col:
        print(f"    Unique donors: {adata.obs[donor_col].nunique()}")
    print(f"  Detected lineage column: {lineage_col}")
    if lineage_col:
        print(f"    Unique lineages: {sorted(adata.obs[lineage_col].unique())}")
    print(f"  Detected tissue column:  {tissue_col}")
    if tissue_col:
        print(f"    Unique tissues: {sorted(adata.obs[tissue_col].unique())}")
    
    # ★ CORE CHECK: Which candidate genes exist in h5ad?
    h5ad_genes = set(adata.var_names)
    found_genes = sorted(ALL_CANDIDATES & h5ad_genes)
    missing_genes = sorted(ALL_CANDIDATES - h5ad_genes)
    
    print(f"\n{'='*50}")
    print(f"  ★ GENE AVAILABILITY RESULT ★")
    print(f"{'='*50}")
    print(f"  Candidates defined:   {len(ALL_CANDIDATES)}")
    print(f"  Found in h5ad:        {len(found_genes)}")
    print(f"  NOT found (missing):  {len(missing_genes)}")
    print(f"\n  Missing genes ({len(missing_genes)}):")
    for g in missing_genes:
        src = 'C3' if g in c3_set else 'C3B-new'
        print(f"    {g:<15} (source: {src})")
    
    print(f"\n  ★★★ ACTUAL ANALYSABLE GENE COUNT = {len(found_genes)} ★★★")

In [ ]:
# ============================================================
# CELL 4: Scan C3 result directory
# ============================================================
print("=" * 70)
print("  STEP 3: C3 result directory contents")
print("=" * 70)

C3_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis/C3_individual_genes/'

# Flexible directory search
BASE = '/content/drive/MyDrive/ITLAS/results/version18-analysis/'
c3_dir_found = None

if os.path.exists(C3_DIR):
    c3_dir_found = C3_DIR
elif os.path.exists(BASE):
    for d in sorted(os.listdir(BASE)):
        if 'c3' in d.lower() or 'individual' in d.lower():
            candidate = os.path.join(BASE, d)
            if os.path.isdir(candidate):
                c3_dir_found = candidate + '/'
                break

if c3_dir_found:
    print(f"✅ C3 directory: {c3_dir_found}")
    files = sorted(os.listdir(c3_dir_found))
    print(f"   {len(files)} files:")
    for f in files[:50]:  # cap at 50
        fpath = os.path.join(c3_dir_found, f)
        sz = os.path.getsize(fpath)
        marker = ' ← GENE LIST' if 'gene_list' in f.lower() else ''
        print(f"   {f} ({sz:,} bytes){marker}")
    if len(files) > 50:
        print(f"   ... and {len(files)-50} more")
else:
    print(f"❌ C3 directory not found.")
    if os.path.exists(BASE):
        print(f"   Available subdirectories:")
        for d in sorted(os.listdir(BASE)):
            if os.path.isdir(os.path.join(BASE, d)):
                nf = len(os.listdir(os.path.join(BASE, d)))
                print(f"     {d}/ ({nf} files)")

In [ ]:
# ============================================================
# CELL 5: Load C3 gene list CSV
# ============================================================
print("=" * 70)
print("  STEP 4: Load C3 gene list CSV")
print("=" * 70)

csv_genes = set()

if c3_dir_found:
    # Find any gene list CSV
    gene_list_files = [f for f in os.listdir(c3_dir_found)
                       if f.endswith('.csv') and 'gene' in f.lower() and 'list' in f.lower()]
    
    if not gene_list_files:
        # Broader search: any CSV with 'gene' or small row count
        gene_list_files = [f for f in os.listdir(c3_dir_found)
                           if f.endswith('.csv') and 'gene' in f.lower()]
    
    print(f"Gene list CSV files found: {gene_list_files}")
    
    for glf in gene_list_files:
        df = safe_read_csv(os.path.join(c3_dir_found, glf), label=glf)
        if df is not None:
            print(f"  Columns: {list(df.columns)}")
            print(f"  First 5 rows:")
            print(df.head().to_string(index=False))
            
            csv_genes = extract_genes_from_df(df, label=glf)
            
            if csv_genes:
                # Cross-reference
                in_csv_not_candidates = csv_genes - ALL_CANDIDATES
                in_candidates_not_csv = ALL_CANDIDATES - csv_genes
                in_h5ad = csv_genes & h5ad_genes if 'h5ad_genes' in dir() else set()
                
                print(f"\n  Cross-reference:")
                print(f"    CSV genes: {len(csv_genes)}")
                print(f"    Overlap with code candidates: {len(csv_genes & ALL_CANDIDATES)}")
                print(f"    In CSV but NOT in code candidates: {len(in_csv_not_candidates)}")
                if in_csv_not_candidates:
                    print(f"      → {sorted(in_csv_not_candidates)}")
                print(f"    In code but NOT in CSV: {len(in_candidates_not_csv)}")
                if in_candidates_not_csv and len(in_candidates_not_csv) < 40:
                    print(f"      → {sorted(in_candidates_not_csv)}")
                if in_h5ad:
                    print(f"    CSV genes found in h5ad: {len(in_h5ad)}")
else:
    print("  (C3 directory not available — skipped)")

In [ ]:
# ============================================================
# CELL 6: Check C5 gene panel
# ============================================================
print("=" * 70)
print("  STEP 5: C5 148-gene panel")
print("=" * 70)

c5_genes = set()
C5_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis/C5_cross_tissue/'

# Flexible search
c5_dir_found = None
if os.path.exists(C5_DIR):
    c5_dir_found = C5_DIR
elif os.path.exists(BASE):
    for d in sorted(os.listdir(BASE)):
        if 'c5' in d.lower() or 'cross' in d.lower():
            candidate = os.path.join(BASE, d)
            if os.path.isdir(candidate):
                c5_dir_found = candidate + '/'
                break

if c5_dir_found:
    print(f"✅ C5 directory: {c5_dir_found}")
    for f in sorted(os.listdir(c5_dir_found)):
        fpath = os.path.join(c5_dir_found, f)
        sz = os.path.getsize(fpath)
        print(f"   {f} ({sz:,} bytes)")
        
        if f.endswith('.csv'):
            df = safe_read_csv(fpath, label=f)
            if df is not None:
                genes = extract_genes_from_df(df, label=f)
                c5_genes.update(genes)
    
    print(f"\n  C5 total unique genes: {len(c5_genes)}")
    
    if c5_genes and 'found_genes' in dir():
        found_set = set(found_genes)
        c3_not_c5 = found_set - c5_genes
        c5_not_c3 = c5_genes - found_set
        print(f"  In C3(h5ad) but NOT C5: {len(c3_not_c5)}")
        if c3_not_c5:
            print(f"    → {sorted(c3_not_c5)}")
        print(f"  In C5 but NOT C3 candidates: {len(c5_not_c3)}")
        if c5_not_c3:
            print(f"    → {sorted(c5_not_c3)}")
else:
    print(f"❌ C5 directory not found.")

In [ ]:
# ============================================================
# CELL 7: Check C4 pathway output (26 vs 29)
# ============================================================
print("=" * 70)
print("  STEP 6: C4 pathway output (26 vs 29?)")
print("=" * 70)

C4_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis/C4_pathway/'

c4_dir_found = None
if os.path.exists(C4_DIR):
    c4_dir_found = C4_DIR
elif os.path.exists(BASE):
    for d in sorted(os.listdir(BASE)):
        if 'c4' in d.lower() or 'pathway' in d.lower():
            candidate = os.path.join(BASE, d)
            if os.path.isdir(candidate):
                c4_dir_found = candidate + '/'
                break

c4_pathways = set()
if c4_dir_found:
    print(f"✅ C4 directory: {c4_dir_found}")
    for f in sorted(os.listdir(c4_dir_found)):
        fpath = os.path.join(c4_dir_found, f)
        sz = os.path.getsize(fpath)
        print(f"   {f} ({sz:,} bytes)")
        
        if f.endswith('.csv'):
            df = safe_read_csv(fpath, label=f)
            if df is not None:
                print(f"    Columns: {list(df.columns)[:10]}")
                pw_col = find_column(df, PATHWAY_COLS)
                if pw_col:
                    pws = sorted(df[pw_col].unique())
                    c4_pathways.update(pws)
                    print(f"    Pathways in '{pw_col}' ({len(pws)}): {pws}")
                else:
                    # Try first string column
                    for col in df.columns:
                        if df[col].dtype == 'object':
                            vals = sorted(df[col].dropna().unique())
                            if 5 < len(vals) < 50:
                                c4_pathways.update(vals)
                                print(f"    Possible pathways in '{col}' ({len(vals)}): {vals}")
                                break
    
    print(f"\n  ★ Total unique pathways found in C4 output: {len(c4_pathways)}")
    if c4_pathways:
        # Check for new 3 pathways
        new3 = {'antigen_presentation', 'type1_ifn', 'tgfb_signaling'}
        found_new = new3 & c4_pathways
        missing_new = new3 - c4_pathways
        print(f"  New 3 pathways present: {sorted(found_new) if found_new else 'NONE'}")
        print(f"  New 3 pathways missing: {sorted(missing_new) if missing_new else 'NONE'}")
else:
    print(f"❌ C4 directory not found.")

In [ ]:
# ============================================================
# CELL 8: Check C9/C9B directory
# ============================================================
print("=" * 70)
print("  STEP 7: C9/C9B output files")
print("=" * 70)

c9_dir_found = None
if os.path.exists(BASE):
    for d in sorted(os.listdir(BASE)):
        if 'c9' in d.lower() or 'verif' in d.lower() or 'fdr' in d.lower():
            candidate = os.path.join(BASE, d)
            if os.path.isdir(candidate):
                c9_dir_found = candidate + '/'
                print(f"✅ C9 directory: {c9_dir_found}")
                for f in sorted(os.listdir(c9_dir_found)):
                    fpath = os.path.join(c9_dir_found, f)
                    sz = os.path.getsize(fpath)
                    marker = ''
                    if 'c3only' in f.lower(): marker = ' ← C3-ONLY GENES'
                    if 'pathway' in f.lower() and 'new' in f.lower(): marker = ' ← NEW PATHWAYS'
                    if 'antigen' in f.lower() or 'ifn' in f.lower(): marker = ' ← NEW PATHWAY'
                    print(f"   {f} ({sz:,} bytes){marker}")
                    
                    # Load C3-only results
                    if 'c3only' in f.lower() and f.endswith('.csv'):
                        df = safe_read_csv(fpath, label=f)
                        if df is not None:
                            print(f"    Columns: {list(df.columns)[:10]}")
                            genes = extract_genes_from_df(df, label=f)
                            if genes:
                                print(f"    C9B C3-only genes: {sorted(genes)}")
                print()

if not c9_dir_found:
    print(f"❌ No C9 directory found.")
    if os.path.exists(BASE):
        print(f"   Available: {[d for d in sorted(os.listdir(BASE)) if os.path.isdir(os.path.join(BASE, d))]}")

In [ ]:
# ============================================================
# CELL 9: Search for C9 new pathway files anywhere
# ============================================================
print("=" * 70)
print("  STEP 8: Search for new pathway (antigen/IFN/TGFB) files")
print("=" * 70)

keywords = ['antigen_pres', 'type1_ifn', 'tgfb_signal', 'new_pathway',
            '3_pathway', 'fix2', '29_pathway', 'c9_pathway']

found_files = []
if os.path.exists(BASE):
    for dirpath, dirnames, filenames in os.walk(BASE):
        for f in filenames:
            f_lower = f.lower()
            if any(kw in f_lower for kw in keywords):
                full = os.path.join(dirpath, f)
                sz = os.path.getsize(full)
                found_files.append(full)
                print(f"  FOUND: {full} ({sz:,} bytes)")

if not found_files:
    print("  No new pathway files found in results tree.")
    print("  → C9 may have stored results within C4 directory or inline in notebook output.")
    print("  → Check C4 directory above: if it has 26 pathways, the 3 new ones are in C9 only.")

In [ ]:
# ============================================================
# CELL 10: ★★★ FINAL SUMMARY ★★★
# ============================================================
print("\n" + "=" * 70)
print("  ★★★ FINAL VERIFICATION SUMMARY ★★★")
print("=" * 70)

print(f"\n[A] CODE-DEFINED CANDIDATES:")
print(f"    C3 unique:        {len(c3_set)}")
print(f"    + C3B truly new:  {len(c3b_truly_new)} ({sorted(c3b_truly_new)})")
print(f"    = Total:          {len(ALL_CANDIDATES)} unique candidates")

if 'found_genes' in dir():
    print(f"\n[B] h5ad FILTERING:")
    print(f"    Found in h5ad:    {len(found_genes)}")
    print(f"    Missing:          {len(missing_genes)} → {sorted(missing_genes)}")
    print(f"    ★ ACTUAL C3 COUNT = {len(found_genes)}")

if csv_genes:
    print(f"\n[C] CSV FILE:")
    print(f"    Genes in CSV:     {len(csv_genes)}")

if c5_genes:
    print(f"\n[D] C5 PANEL:")
    print(f"    C5 genes:         {len(c5_genes)}")
    if 'found_genes' in dir():
        diff = set(found_genes) - c5_genes
        print(f"    C3 minus C5:      {len(diff)} genes dropped")

if c4_pathways:
    print(f"\n[E] C4 PATHWAYS:")
    print(f"    In C4 output:     {len(c4_pathways)} pathways")

print(f"\n[F] NUMBER RECONCILIATION:")
print(f"    Notebook title says '169'")
print(f"    Methods v3.1 says '173'")
print(f"    CSV filename says '196'")
if 'found_genes' in dir():
    print(f"    ★ h5ad verification = {len(found_genes)} ← USE THIS NUMBER")

print(f"\n[G] MANUSCRIPT RECOMMENDATION:")
if 'found_genes' in dir():
    print(f"    '{len(ALL_CANDIDATES)} candidate genes were curated from")
    print(f"     published immune signatures; {len(found_genes)} were detected")
    print(f"     in the dataset and included in individual gene analysis.'")

print("\n" + "=" * 70)
print("  COPY ALL OUTPUT ABOVE FOR MANUSCRIPT CORRECTION")
print("=" * 70)